# A05. 미사용/미재사용 원인 분석

가설:
- 초기 답변불가 경험
- 여신고객 비중
- 탈회/직군 분포

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

root = Path.cwd()
refactor_dir = root if root.name == 'refactor' else (root / 'analysis' / 'refactor')
if not refactor_dir.exists():
    refactor_dir = Path('/workspace/analysis/refactor')
if str(refactor_dir) not in sys.path:
    sys.path.insert(0, str(refactor_dir))

import common
import a05_nonreuse_causes as a05
print('refactor_dir =', refactor_dir)

In [ ]:
PROFILE_FILE = '/home/jovyan/cjs/refactor2/data/user.csv'
CHAT_FILE = '/home/jovyan/cjs/refactor2/data/chat.csv'
NONREUSE_MAX_REQ = 2

profile = common.normalize_profile(common.load_csv(PROFILE_FILE))
chat = common.normalize_chat(common.load_csv(CHAT_FILE))

non_label, reuse_label = a05.group_labels(NONREUSE_MAX_REQ)
reuse = a05.build_reuse_group(profile, chat, NONREUSE_MAX_REQ)
unans = a05.build_unanswered_features(chat)

base = profile.merge(reuse, on='customer_id', how='inner').merge(unans, on='customer_id', how='left')
t_n1 = a05.test_n1(base, non_label, reuse_label)
t_n2 = a05.test_n2(base, non_label, reuse_label)
t_n3 = a05.test_n3(base, non_label, reuse_label)
result = pd.concat([t_n1, t_n2, t_n3], ignore_index=True)
result

In [ ]:
plot = result.dropna(subset=['diff_nonreuse_minus_reuse']).copy()
if not plot.empty:
    plt.figure(figsize=(8,4))
    sns.barplot(data=plot.sort_values('diff_nonreuse_minus_reuse', ascending=False), x='diff_nonreuse_minus_reuse', y='metric', palette='mako')
    plt.axvline(0, color='black', linewidth=1)
    plt.title('미재사용군 - 재사용군 차이')
    plt.tight_layout()
    plt.show()